# Phase 10 — Heterogeneity

Tests whether the arm contrast varies across four pre-declared splits: career
stage, country income group, subject domain, and detection lag.

**Input:** `data/interim/phase08_panel_extended.csv`

**Output:** `data/results/phase10_heterogeneity.csv`

The quantity tested is the difference in the arm contrast between subgroups, not
each subgroup's contrast on its own, so the difference carries its own standard
error. Subgroups partition the sample, so the SE of a between-subgroup
difference is the root sum of squares; within a subgroup, two arms come from one
regression. A placebo at a fake event runs inside every subgroup.

In [1]:
import os
import math
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

PANEL = "data/interim/phase08_panel_extended.csv"
OUT_RESULTS = "data/results/phase10_heterogeneity.csv"

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

# Omitted arm; contrasts are each arm minus this one, from one regression.
REF_ARM = "EDITORIAL_COMPROMISE"

REF_EVENTS = (-2, -1)
ANALYSIS_PRE, ANALYSIS_POST = 3, 6
PLACEBO_OFFSET = -4

SPLITS = ["career_band", "income_group", "subject_group", "lag_band"]
OUTCOMES = ["active", "publications"]
HORIZONS = (1, 3, 6)

CLUSTER_VAR = "rw_journal"
YEAR_FE = False

MIN_AUTHORS_PER_SUBGROUP = 100

pd.set_option("display.width", 220)
os.makedirs("data/results", exist_ok=True)

print(f"reference arm     {REF_ARM}")
print(f"splits            {SPLITS}")
print(f"horizons          {HORIZONS}")

reference arm     EDITORIAL_COMPROMISE
splits            ['career_band', 'income_group', 'subject_group', 'lag_band']
horizons          (1, 3, 6)


## Estimation engine

In [2]:
def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def absorb_fe(df, cols, groups, tol=1e-10, max_iter=100):
    out = df[cols].astype(float).copy()
    for _ in range(max_iter):
        prev = out.values.copy()
        for g in groups:
            gg = df[g]
            for c in cols:
                out[c] = out[c] - out[c].groupby(gg).transform("mean")
        if np.abs(out.values - prev).max() < tol:
            break
    return out


def ols_cluster(y, X, cluster, names):
    y = np.asarray(y, float); X = np.asarray(X, float)
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta

    cl = pd.Series(cluster).astype(str).values
    order = np.argsort(cl)
    cl_s, X_s, r_s = cl[order], X[order], resid[order]
    bounds = np.flatnonzero(np.r_[True, cl_s[1:] != cl_s[:-1], True])
    meat = np.zeros((k, k))
    for a, b in zip(bounds[:-1], bounds[1:]):
        s = X_s[a:b].T @ r_s[a:b]
        meat += np.outer(s, s)

    G = len(bounds) - 1
    dof = (G / max(G - 1, 1)) * ((n - 1) / max(n - k, 1))
    V = dof * (XtX_inv @ meat @ XtX_inv)
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(se > 0, beta / se, np.nan)
    p = np.array([2 * (1 - _norm_cdf(abs(ti))) if np.isfinite(ti) else np.nan
                  for ti in t])
    res = pd.DataFrame({"term": names, "coef": beta, "se": se, "p": p})
    res.attrs["n_clusters"] = G
    return res


def check_design(X):
    if np.linalg.matrix_rank(X) < X.shape[1]:
        return False, "rank-deficient", np.inf
    c = float(np.linalg.cond(X.T @ X))
    return (c <= 1e10), ("" if c <= 1e10 else "ill-conditioned"), c


def arm_contrasts(panel, outcome, pre=None, post=None):
    """Each arm minus REF_ARM at each event time, from one regression.

    REF_ARM is the omitted category, so a coefficient is the contrast against it
    with a covariance-correct standard error. Event dummies alone carry the
    common path; the interactions carry the contrast.
    """
    pre = ANALYSIS_PRE if pre is None else pre
    post = ANALYSIS_POST if post is None else post

    d = panel.dropna(subset=[outcome, CLUSTER_VAR, "arm"]).copy()
    d = d[(d.event_time >= -pre) & (d.event_time <= post)]
    d = d[d.arm.isin(ARMS)]
    if d.empty or d.arm.nunique() < 2:
        return None

    others = [a for a in ARMS if a != REF_ARM and a in set(d.arm)]
    if not others:
        return None

    ks = [k for k in sorted(d.event_time.unique()) if k not in REF_EVENTS]
    ev = pd.DataFrame({f"k{k:+d}": (d.event_time == k).astype(float)
                       for k in ks}, index=d.index)

    parts, names = [], []
    parts.append(ev.values)
    names += [f"{c}:BASE" for c in ev.columns]
    for a in others:
        m = (d.arm == a).astype(float).values[:, None]
        parts.append(ev.values * m)
        names += [f"{c}:{a}" for c in ev.columns]
    X = np.hstack(parts)
    y = d[outcome].values.astype(float)

    fe = ["author_id"] + (["year"] if YEAR_FE else [])
    tmp = pd.DataFrame(X, columns=[f"c{i}" for i in range(X.shape[1])],
                       index=d.index)
    tmp[outcome] = y
    for g in fe:
        tmp[g] = d[g].values
    ab = absorb_fe(tmp, [f"c{i}" for i in range(X.shape[1])] + [outcome], fe)
    X, y = ab[[f"c{i}" for i in range(X.shape[1])]].values, ab[outcome].values

    ok, why, cond = check_design(X)
    if not ok:
        return None

    res = ols_cluster(y, X, d[CLUSTER_VAR].values, names)
    parsed = res.term.str.extract(r"^k([+-]?\d+):(.+)$")
    res["event_time"] = pd.to_numeric(parsed[0], errors="coerce")
    res["arm"] = parsed[1]
    res = res[res.arm != "BASE"].dropna(subset=["event_time"])
    res["n_authors"] = d.author_id.nunique()
    res["n_obs"] = len(d)
    return res

## Load

In [3]:
panel = pd.read_csv(PANEL, low_memory=False)
full = panel[panel.balanced & panel.in_primary].copy()
est = full[(full.event_time >= -ANALYSIS_PRE) & (full.event_time <= ANALYSIS_POST)]

print(f"panel             {len(panel):,} rows")
print(f"estimation sample {est.author_id.nunique():,} authors, "
      f"event time -{ANALYSIS_PRE}..+{ANALYSIS_POST}")
print(est.groupby("author_id").arm.first().value_counts().to_string())

panel             706,946 rows
estimation sample 19,265 authors, event time -3..+6
arm
AUTHOR_MISCONDUCT       9617
HONEST_ERROR            4894
EDITORIAL_COMPROMISE    2783
UNCONFIRMED_CONCERNS    1457
ETHICS_VIOLATION         327
UNCLASSIFIED             187


## Contrasts within each subgroup

In [4]:
per_group = []

for split in SPLITS:
    if split not in est.columns or est[split].isna().all():
        continue
    for level, sub in est.groupby(split, observed=True):
        n = sub.author_id.nunique()
        if n < MIN_AUTHORS_PER_SUBGROUP:
            print(f"  {split}={level}: {n:,} authors, below "
                  f"{MIN_AUTHORS_PER_SUBGROUP}; skipped")
            continue
        for outcome in OUTCOMES:
            if outcome not in sub.columns:
                continue
            r = arm_contrasts(sub, outcome)
            if r is None:
                continue
            r["split"] = split
            r["subgroup"] = str(level)
            r["outcome"] = outcome
            per_group.append(r)

per_group = (pd.concat(per_group, ignore_index=True)
             if per_group else pd.DataFrame())
print(f"subgroup contrasts estimated: {len(per_group):,}")

  subject_group=OTHER: 46 authors, below 100; skipped
subgroup contrasts estimated: 416


## The heterogeneity test

In [5]:
def norm_p(z):
    return 2 * (1 - _norm_cdf(abs(z))) if np.isfinite(z) else np.nan


het = []
if not per_group.empty:
    for (outcome, split, arm, k), grp in per_group.groupby(
            ["outcome", "split", "arm", "event_time"], observed=True):
        grp = grp.sort_values("subgroup")
        if len(grp) < 2:
            continue
        base = grp.iloc[0]
        for _, other in grp.iloc[1:].iterrows():
            diff = float(other.coef - base.coef)
            se = float(np.hypot(other.se, base.se))
            z = diff / se if se > 0 else np.nan
            het.append({
                "outcome": outcome, "split": split, "arm": arm,
                "event_time": int(k),
                "subgroup": other.subgroup, "vs_subgroup": base.subgroup,
                "diff_in_diff": round(diff, 4), "se": round(se, 4),
                "z": round(z, 3) if np.isfinite(z) else np.nan,
                "p": round(norm_p(z), 4) if np.isfinite(z) else np.nan,
            })

het = pd.DataFrame(het)
if not het.empty:
    n_sig = int((het.p < 0.05).sum())
    exp = 0.05 * len(het)
    print(f"heterogeneity tests: {len(het):,}")
    print(f"significant at 5%:   {n_sig:,}")
    print(f"expected by chance:  {exp:.0f}")
else:
    print("no heterogeneity tests")

heterogeneity tests: 320
significant at 5%:   17
expected by chance:  16


## Placebo within each subgroup

The same contrasts at a fake event four years earlier. A subgroup effect no
larger than its placebo is trend rather than treatment.

In [6]:
plac = full.copy()
plac["event_time"] = plac.event_time - PLACEBO_OFFSET
plac = plac[(plac.event_time >= -ANALYSIS_PRE) & (plac.event_time <= 3)]

placebo = []
for split in SPLITS:
    if split not in plac.columns or plac[split].isna().all():
        continue
    for level, sub in plac.groupby(split, observed=True):
        if sub.author_id.nunique() < MIN_AUTHORS_PER_SUBGROUP:
            continue
        for outcome in OUTCOMES:
            if outcome not in sub.columns:
                continue
            r = arm_contrasts(sub, outcome, pre=ANALYSIS_PRE, post=3)
            if r is None:
                continue
            r["split"] = split
            r["subgroup"] = str(level)
            r["outcome"] = outcome
            placebo.append(r)

placebo = pd.concat(placebo, ignore_index=True) if placebo else pd.DataFrame()
print(f"placebo contrasts estimated: {len(placebo):,}")

if not placebo.empty and not per_group.empty:
    key = ["outcome", "split", "subgroup", "arm", "event_time"]
    comp = (per_group[key + ["coef"]].rename(columns={"coef": "real"})
            .merge(placebo[key + ["coef"]].rename(columns={"coef": "placebo"}),
                   on=key, how="inner"))
    comp = comp[comp.event_time.isin([1, 2, 3])]
    comp["ratio"] = (comp.real.abs() /
                     comp.placebo.abs().replace(0, np.nan)).round(2)
    failed = comp[comp.real.abs() <= comp.placebo.abs()]

    print(f"contrasts comparable at both events: {len(comp):,}")
    print(f"  no larger than their placebo:      {len(failed):,}")

placebo contrasts estimated: 260
contrasts comparable at both events: 156
  no larger than their placebo:      92


## Write

In [7]:
out = []
if not per_group.empty:
    out.append(per_group.assign(table="subgroup_contrast"))
if not het.empty:
    out.append(het.assign(table="heterogeneity_test"))
if not placebo.empty:
    out.append(placebo.assign(table="placebo"))

if out:
    pd.concat(out, ignore_index=True).to_csv(OUT_RESULTS, index=False)
    print(f"{OUT_RESULTS}")
    for t in out:
        print(f"  {t.table.iloc[0]:<22} {len(t):,} rows")
else:
    print("nothing estimated")

data/results/phase10_heterogeneity.csv
  subgroup_contrast      416 rows
  heterogeneity_test     320 rows
  placebo                260 rows
